Use your implementation of Gradient Descent from Homework 2 and adapt it for logistic regression. Take 3 values of the learning rate and report
the cross-entropy loss objective after 10, 50, and 100 iterations. At 100 iterations, report the accuracy, precision, recall, and F1 score for the 3 learning rates, and compare with the metrics given by the package on the training and testing sets.

In [78]:
import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score

In [3]:
# import fetch dataset 
spambase = fetch_ucirepo(id=94) 
  
# data (as pandas dataframes) 
X = spambase.data.features 
y = spambase.data.targets 
  
# metadata 
#print(spambase.metadata) 
  
# variable information 
#print(spambase.variables) 

In [72]:
# split data into training and test sets:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, shuffle=True) 

In [73]:
# scale the data: 

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [74]:
# gradient descent implementation from HW2 adapted for logistic regression

def logistic_regression(features, labels, alpha, max_iterations):

    # convert everything to numpy:
    features = np.array(features)
    labels = np.array(labels).flatten()
    
    # initialize theta to 0
    theta = np.zeros(features.shape[1])
    N = features.shape[0]

    for iteration in range(max_iterations):
        # 1. get predicted values -> changed to adapt to logistic regression
        # 1/(1 + e^-(theta.T x))
        y_pred = 1 / (1 + (np.exp(-(features @ theta))))

        # 2. find error difference btwn predicted and actual
        error = y_pred - labels

        # 3. gradient: 1/N * X.T * (error calculated above) 
        gradient = (1/N) * (features.T @ error)

        # 4. update:
        theta = theta - (alpha * gradient)

    return theta

In [75]:
# sigmoid function converts output of model into probability value between 0 and 1
def sigmoid(features, theta):
    return 1 / (1 + (np.exp(-(features @ theta))))

In [76]:
# should result in class label 0 or 1
def predicted_y(features, theta):
    features = np.array(features)
    return (sigmoid(features, theta) >= 0.5).astype(int)

In [77]:
# cross entropy loss objective (minimize): 
# J(theta) = -sum[yi*log(h0(xi)) + (1-yi)log(1-h0(xi))]
def cross_entropy_loss(features, labels, theta):
    features = np.array(features)
    labels = np.array(labels).flatten()

    N = features.shape[0]
    y_pred = sigmoid(features, theta) # y_pred is a probability value
    y_pred = np.clip(y_pred, 1e-10, 1 - 1e-10) # avoid log(0)
    return -(1/N) * np.sum(labels * np.log(y_pred) + (1-labels)* np.log(1-y_pred))

In [79]:
# take 3 values of the learning rate and report the cross-entropy loss objective after 
# 10, 50 and 100 iterations

max_iterations = [10, 50, 100]
alphas = [0.01, 0.1, 0.5] # 3 values of learning rates
losses = []

for alpha in alphas:
    loss1 = []
    for iteration in max_iterations:

        theta = logistic_regression(X_train_scaled, y_train, alpha, iteration)
        loss = cross_entropy_loss(X_train_scaled, y_train, theta)

        loss1.append(loss)
    
    losses.append(loss1)

In [81]:
# organize all variables into a table
data = []

for alpha in range(0, len(alphas)):
    
    stats_train = { 
        'alpha': alphas[alpha],
        'Iterations': max_iterations,
        'Cross Entropy Loss': losses[alpha]
    }

    data.append(pd.DataFrame(stats_train))

print("Cross Entropy Loss Objective After 10, 50, and 100 Iterations for Different Learning Rates (Alpha)")
data

Cross Entropy Loss Objective After 10, 50, and 100 Iterations for Different Learning Rates (Alpha)


[   alpha  Iterations  Cross Entropy Loss
 0   0.01          10            0.652396
 1   0.01          50            0.547160
 2   0.01         100            0.478240,
    alpha  Iterations  Cross Entropy Loss
 0    0.1          10            0.474978
 1    0.1          50            0.340830
 2    0.1         100            0.304437,
    alpha  Iterations  Cross Entropy Loss
 0    0.5          10            0.337069
 1    0.5          50            0.271804
 2    0.5         100            0.257045]

In [90]:
# At 100 iterations, report the accuracy, precision, recall, and F1 score for the 3 learning rates, 
# and compare with the metrics given by the package on the training and testing sets.

train_rows = []
test_rows = []

for alpha in alphas:
    
    theta  = logistic_regression(X_train_scaled, y_train, alpha, 100)

    y_pred_train = predicted_y(X_train_scaled, theta)
    y_pred_test = predicted_y(X_test_scaled, theta)

    train_rows.append({
        "Alpha": alpha,
        "Accuracy": accuracy_score(y_train, y_pred_train),
        "Precision": precision_score(y_train, y_pred_train),
        "Recall": recall_score(y_train, y_pred_train), 
        "F1": f1_score(y_train, y_pred_train)
    })

    test_rows.append({
        "Alpha": alpha,
        "Accuracy": accuracy_score(y_test, y_pred_test),
        "Precision": precision_score(y_test, y_pred_test),
        "Recall": recall_score(y_test, y_pred_test), 
        "F1": f1_score(y_test, y_pred_test)
    })


print("Scores for 3 Learning Rates with Manual Logistic Regression For Training Set")
display(pd.DataFrame(train_rows).set_index("Alpha"))

print("\nScores for 3 Learning Rates with Manual Logistic Regression For Test Set")
display(pd.DataFrame(test_rows).set_index("Alpha"))

Scores for 3 Learning Rates with Manual Logistic Regression For Training Set


,Accuracy,Precision,Recall,F1
Alpha,,,,
0.01,0.892754,0.840366,0.893124,0.865942
0.10,0.911884,0.885246,0.887892,0.886567
0.50,0.919420,0.908951,0.880419,0.894457



Scores for 3 Learning Rates with Manual Logistic Regression For Test Set


,Accuracy,Precision,Recall,F1
Alpha,,,,
0.01,0.899218,0.861167,0.901053,0.880658
0.10,0.907906,0.895075,0.880000,0.887473
0.50,0.914857,0.921700,0.867368,0.893709


In [94]:
# metrics from pacakge Logistic Regression model:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred_test = model.predict(X_test_scaled)
y_pred_train = model.predict(X_train_scaled)

train_rows = []
test_rows = []

#print("Test Set Scores ")
test_rows.append({
    "Model": "Package Model",
    "Accuracy": accuracy_score(y_test, y_pred_test),
    "Precision": precision_score(y_test, y_pred_test),
    "Recall": recall_score(y_test, y_pred_test),
    "F1": f1_score(y_test, y_pred_test)
})

#print("\nTrain Set Scores - sklearn")
train_rows.append({
    "Model": "Package Model",
    "Accuracy": accuracy_score(y_train, y_pred_train),
    "Precision": precision_score(y_train, y_pred_train),
    "Recall": recall_score(y_train, y_pred_train),
    "F1": f1_score(y_train, y_pred_train)
})

print("Scores for 3 Learning Rates with Package Logistic Regression For Training Set")
display(pd.DataFrame(train_rows).set_index("Model"))

print("\nScores for 3 Learning Rates with Package Logistic Regression For Test Set")
display(pd.DataFrame(test_rows).set_index("Model"))


/Users/pavithra/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:1184: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Scores for 3 Learning Rates with Package Logistic Regression For Training Set


/Users/pavithra/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/pavithra/anaconda3/lib/python3.11/site-packages/sklearn/base.py:464: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/Users/pavithra/anaconda3/lib/python3.11/site-packages/sklearn/base.py:464: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


,Accuracy,Precision,Recall,F1
Model,,,,
Package Model,0.89971,0.825887,0.939462,0.879021



Scores for 3 Learning Rates with Package Logistic Regression For Test Set


,Accuracy,Precision,Recall,F1
Model,,,,
Package Model,0.914857,0.859048,0.949474,0.902
